## Laboratorio Neo4j - CRUD y Algoritmos de Grafos

## Sección 1 - Creación de la conexión

Edita tu URI y credenciales de conexión en la celda de abajo. Usaremos el driver oficial de Python `neo4j`.

In [5]:
neo4j_uri = "bolt://localhost:7687"
neo4j_user = "neo4j"
neo4j_password = "password"

In [6]:
from neo4j import GraphDatabase

try:
    driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
    driver.verify_connectivity()
    print("✅ Conexión exitosa a Neo4j")
except Exception as e:
    print(f"❌ Error conectando a Neo4j: {e}")

✅ Conexión exitosa a Neo4j


## Sección 2 - Inserción de datos (CREATE)

Vamos a inicializar nuestra base de datos con el ecosistema de *GraphStore*. 

Utiliza el método `session.execute_write()` para ejecutar una consulta Cypher que inserte el siguiente lote de datos usando `UNWIND` y `MERGE`:

```json
[
  {cliente: "Alice", producto: "Laptop", tarjeta: "1111-2222"},
  {cliente: "Bob", producto: "Laptop", tarjeta: "3333-4444"},
  {cliente: "Bob", producto: "Mouse", tarjeta: "3333-4444"},
  {cliente: "Charlie", producto: "Mouse", tarjeta: "1111-2222"},
  {cliente: "Diana", producto: "Teclado", tarjeta: "5555-6666"},
  {cliente: "Alice", producto: "Audífonos", tarjeta: "1111-2222"}
]
```

Asegúrate de crear los nodos `Usuario`, `Producto`, `Tarjeta` y las relaciones `[:COMPRÓ]` y `[:USA]`.

In [30]:
data = [
    {"cliente": "Alice",   "producto": "Laptop",    "tarjeta": "1111-2222"},
    {"cliente": "Bob",     "producto": "Laptop",    "tarjeta": "3333-4444"},
    {"cliente": "Bob",     "producto": "Mouse",     "tarjeta": "3333-4444"},
    {"cliente": "Charlie", "producto": "Mouse",     "tarjeta": "1111-2222"},
    {"cliente": "Diana",   "producto": "Teclado",   "tarjeta": "5555-6666"},
    {"cliente": "Alice",   "producto": "Audífonos", "tarjeta": "1111-2222"},
]

def insert_data(tx, batch):
    tx.run("""
        UNWIND $batch AS row
        MERGE (u:Usuario {nombre: row.cliente}) 
        MERGE (p:Producto {nombre: row.producto})
        MERGE (t:Tarjeta  {numero: row.tarjeta})
        MERGE (u)-[:COMPRÓ]->(p)
        MERGE (u)-[:USA]->(t)
    """, batch=batch)

#Usamos MERGE para evitar duplicados, y UNWIND para insertar en batch
# UNWIND $batch AS row es equivalente a un for row in batch
# Sin UNWIND tendrías que hacer una llamada al driver por cada elemento — mucho más lento.


with driver.session() as session:
    session.execute_write(insert_data, data)
    print("Datos insertados correctamente")

Datos insertados correctamente


In [8]:
print("Validando grafo insertado...")

Validando grafo insertado...


## Sección 3 - Consultas (READ)

Escribe una consulta Cypher para encontrar todos los productos que compró "Bob".

***Almacena el resultado (una lista con los nombres de los productos) en una variable llamada `productos_bob`***

In [34]:
productos_bob = []

def get_productos_bob(tx):
    result = tx.run("""
        MATCH (u:Usuario {nombre: 'Bob'})-[:COMPRÓ]->(p:Producto)
        RETURN p.nombre AS producto
    """)
    return [record["producto"] for record in result]

with driver.session() as session:
    productos_bob = session.execute_read(get_productos_bob)

print(productos_bob)

['Mouse', 'Laptop']


In [11]:
print('Validando resultado de la consulta en la variable:', productos_bob)

Validando resultado de la consulta en la variable: ['Mouse', 'Laptop']


## Sección 4 - Actualización (UPDATE)

Diana ha reportado que perdió su tarjeta y el banco le ha enviado una nueva.
Actualiza el nodo de la tarjeta que Diana `[:USA]` para que su número ahora sea `'9999-9999'`.

In [35]:
def update_tarjeta_diana(tx):
    tx.run("""
        MATCH (u:Usuario {nombre: 'Diana'})-[:USA]->(t:Tarjeta)
        SET t.numero = '9999-9999'
    """)
with driver.session() as session:
    session.execute_write(update_tarjeta_diana)
    print("Tarjeta de Diana actualizada")

Tarjeta de Diana actualizada


In [13]:
print("Validando actualización...")

Validando actualización...


## Sección 5 - El Poder de los Grafos: Algoritmos y Patrones

Aquí es donde Neo4j y Cypher demuestran su superioridad sobre las bases de datos tabulares. Vamos a resolver problemas complejos de conectividad.

### 5.1 Motor de Recomendación Colaborativo

Alice acaba de entrar a la tienda. Busca qué otros usuarios compraron los mismos productos que Alice y **recomiéndale** los productos que esos otros usuarios compraron, pero que Alice aún NO tiene.

***Guarda el nombre del producto recomendado en una variable llamada `recomendacion_alice`***

In [36]:
recomendacion_alice = None
def get_recomendacion_alice(tx):
    result = tx.run("""
        MATCH (alice:Usuario {nombre: 'Alice'})-[:COMPRÓ]->(p:Producto)<-[:COMPRÓ]-(otro:Usuario)
        MATCH (otro)-[:COMPRÓ]->(recomendado:Producto)
        WHERE NOT (alice)-[:COMPRÓ]->(recomendado)
          AND recomendado <> p
        RETURN DISTINCT recomendado.nombre AS recomendacion
    """)
    record = result.single()
    return record["recomendacion"] if record else None

with driver.session() as session:
    recomendacion_alice = session.execute_read(get_recomendacion_alice)

print(f"Recomendación para Alice: {recomendacion_alice}")

Recomendación para Alice: Mouse



    Alice ──COMPRÓ──> [Laptop] <──COMPRÓ── Bob

                                            └──COMPRÓ──> Mouse  ← recomendar esto


In [15]:
print("Validando el motor de recomendaciones...")

Validando el motor de recomendaciones...


### 5.2 Búsqueda de Rutas (Shortest Path)

¿A cuántos 'grados de separación' están conectados Diana y Charlie? Utiliza la función `shortestPath()` para encontrar el camino más corto entre ellos sin importar el tipo de relación o la dirección de la flecha.

***Almacena el número de saltos (longitud del camino) en una variable `distancia_diana_charlie`.***

In [37]:
distancia_diana_charlie = 0
def get_distancia_diana_charlie(tx):
    result = tx.run("""
        MATCH (diana:Usuario {nombre: 'Diana'}), (charlie:Usuario {nombre: 'Charlie'})
        MATCH path = shortestPath((diana)-[*]-(charlie))
        RETURN length(path) AS distancia
    """)
    record = result.single()
    return record["distancia"] if record else None

with driver.session() as session:
    distancia_diana_charlie = session.execute_read(get_distancia_diana_charlie)

print(f"Grados de separación: {distancia_diana_charlie}")

Grados de separación: None


shortestPath() → Es una función nativa de Neo4j que implementa BFS (Breadth-First Search) internamente. Al ser nativa, está optimizada en Java y es mucho más eficiente que escribir la búsqueda manualmente.

[*] → cualquier tipo de relación, cualquier dirección, cualquier cantidad de saltos

length(path) → cuenta el número de relaciones (saltos) del camino


In [18]:
print("Validando cálculo de ruta más corta...")

Validando cálculo de ruta más corta...


### 5.3 Detección de Fraude (Patrones de Anomalía)

El equipo de seguridad sospecha de tarjetas clonadas. Encuentra si existen **dos usuarios diferentes** que estén compartiendo exactamente la **misma tarjeta de crédito** en el sistema.

***Almacena el resultado (una lista de tuplas con los nombres de los usuarios sospechosos, ej: `[('Alice', 'Charlie')]`) en la variable `usuarios_sospechosos`.***
*Nota: Para evitar duplicados como (A, B) y (B, A), recuerda usar `id(u1) < id(u2)` o comparar sus nombres alfabéticamente.*

In [38]:
usuarios_sospechosos = []
def get_usuarios_sospechosos(tx):
    result = tx.run("""
        MATCH (u1:Usuario)-[:USA]->(t:Tarjeta)<-[:USA]-(u2:Usuario)
        WHERE u1.nombre < u2.nombre
        RETURN u1.nombre AS usuario1, u2.nombre AS usuario2
    """)
    return [(r["usuario1"], r["usuario2"]) for r in result]

with driver.session() as session:
    usuarios_sospechosos = session.execute_read(get_usuarios_sospechosos)

print(f"Usuarios sospechosos encontrados: {usuarios_sospechosos}")

Usuarios sospechosos encontrados: [('Alice', 'Charlie')]


In [21]:
print("Validando algoritmo de detección de fraude...")

Validando algoritmo de detección de fraude...


## Sección 6 - Eliminación de datos (DELETE)

Alice ha decidido devolver los 'Audífonos' a la tienda. Escribe una consulta para eliminar **únicamente la relación** `[:COMPRÓ]` entre Alice y los Audífonos, sin borrar los nodos de Usuario ni de Producto.

***Guarda el valor `True` en la variable `relacion_eliminada` si el código se ejecuta sin errores.***

In [39]:
relacion_eliminada = False

def delete_relacion(tx):
    tx.run("""
        MATCH (u:Usuario {nombre: 'Alice'})-[r:COMPRÓ]->(p:Producto {nombre: 'Audífonos'})
        DELETE r
    """)

with driver.session() as session:
    session.execute_write(delete_relacion)
    relacion_eliminada = True  #No podemos validar directamente que la relación se eliminó, pero al menos confirmamos que el código se ejecutó sin errores.

print(f"Relación eliminada: {relacion_eliminada}")

Relación eliminada: True


In [40]:
print("Validando eliminación de relación...")
productos_alice = []

def get_productos_alice(tx):
    result = tx.run("""
        MATCH (u:Usuario {nombre: 'Alice'})-[:COMPRÓ]->(p:Producto)
        RETURN p.nombre AS producto
    """)
    return [record["producto"] for record in result]

with driver.session() as session:
    productos_alice = session.execute_read(get_productos_alice)

print(productos_alice)

Validando eliminación de relación...
['Laptop']


## Sección 7 - Agrupación y Ordenamiento (WITH, ORDER BY)

¿Cuáles son los productos más populares en la tienda? Cuenta el número de usuarios que compraron cada producto y filtra para obtener los productos que tienen **más de 1 venta**.

***Guarda el resultado (una lista con los nombres de los productos) en `productos_populares`.***

In [41]:
productos_populares = []
def get_productos_populares(tx):
    result = tx.run("""
        MATCH (p:Producto)<-[:COMPRÓ]-(u:Usuario)
        WITH p, count(u) AS ventas
        WHERE ventas > 1
        RETURN p.nombre AS producto
    """)
    return [record["producto"] for record in result]

with driver.session() as session:
    productos_populares = session.execute_read(get_productos_populares)

print(f"Productos con más de 1 venta: {productos_populares}")

Productos con más de 1 venta: ['Laptop', 'Mouse']


In [27]:
print("Validando agrupación y ordenamiento...")

Validando agrupación y ordenamiento...


## Sección 8 - Filtrado por Patrones Indirectos (Co-compras)

Queremos sugerir accesorios directamente en la página de ventas de la 'Laptop'. Encuentra qué *otros productos* compraron los usuarios que también adquirieron una 'Laptop' (excluyendo la 'Laptop' en los resultados finales).

***Guarda el resultado (una lista con los nombres de estos productos) en `productos_co_comprados`.***

In [42]:
productos_co_comprados = []
def get_co_compras(tx):
    result = tx.run("""
        MATCH (u:Usuario)-[:COMPRÓ]->(laptop:Producto {nombre: 'Laptop'})
        MATCH (u)-[:COMPRÓ]->(otro:Producto)
        WHERE otro.nombre <> 'Laptop'
        RETURN DISTINCT otro.nombre AS producto
    """)
    return [r["producto"] for r in result]

with driver.session() as session:
    productos_co_comprados = session.execute_read(get_co_compras)
    
print(f"Quienes compraron Laptop también compraron: {productos_co_comprados}")

Quienes compraron Laptop también compraron: ['Mouse']


In [29]:
print("Validando patrones indirectos...")

Validando patrones indirectos...


## Sección 9 - Centralidad de Grado (Degree Centrality)

Encuentra al usuario que ha realizado la mayor cantidad de compras en toda la tienda (el nodo de tipo Usuario con más relaciones salientes de tipo `[:COMPRÓ]`).

***Guarda el nombre de este usuario en la variable de texto `usuario_mas_activo`.***

In [43]:
usuario_mas_activo = ""
def get_usuario_mas_activo(tx):
    result = tx.run("""
        MATCH (u:Usuario)-[:COMPRÓ]->(p:Producto)
        WITH u.nombre AS usuario, count(p) AS compras
        ORDER BY compras DESC
        LIMIT 1
        RETURN usuario
    """)
    record = result.single()
    return record["usuario"] if record else None

with driver.session() as session:
    usuario_mas_activo = session.execute_read(get_usuario_mas_activo)

print(f"El usuario más activo es: {usuario_mas_activo}")

El usuario más activo es: Bob


In [33]:
print("Validando algoritmo de centralidad...")

Validando algoritmo de centralidad...
